In [ ]:
import os

print("Contents of /kaggle/input:")
for d in os.listdir("/kaggle/input"):
    print(" -", d)
    full_path = os.path.join("/kaggle/input", d)
    if os.path.isdir(full_path):
        sub = os.listdir(full_path)
        print("    subfolders/files:", sub[:10])

In [6]:
import os
import subprocess
import sys

def pip(args):
    result = subprocess.run(["pip"] + args, capture_output=True, text=True)
    if result.returncode != 0:
        print("PIP STEP FAILED:", args)
        print(result.stderr[-1500:])
    return result.returncode

# clone vesselFM if it's not already sitting in /kaggle/working
if not os.path.exists("/kaggle/working/vesselFM"):
    print("Cloning vesselFM...")
    ret = os.system("git clone --quiet https://github.com/bwittmann/vesselFM /kaggle/working/vesselFM")
    if ret != 0:
        raise RuntimeError("git clone failed -- check network access in notebook settings.")
else:
    print("vesselFM folder already present.")

config_path = "/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml"
assert os.path.exists(config_path), f"Config missing: {config_path}"

# install what's needed, skip vesselFM's own requirements.txt so torch stays untouched
pip(["install", "-q", "monai", "hydra-core", "omegaconf", "huggingface_hub", "scikit-image", "nibabel", "tqdm"])
pip(["install", "-q", "-e", "/kaggle/working/vesselFM", "--no-deps"])
pip(["install", "-q", "--force-reinstall", "numpy==1.26.4"])

# load the model in this same cell, same kernel, right after the numpy pin
import torch
print("\ntorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

sys.path.insert(0, "/kaggle/working/vesselFM")
import hydra
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_cfg = OmegaConf.load(config_path)
ckpt_path = hf_hub_download(repo_id="bwittmann/vesselFM", filename="vesselFM_base.pt")
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)

model = hydra.utils.instantiate(model_cfg)
model.load_state_dict(ckpt)
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"\nvesselFM loaded on {device}. Total params: {n_params/1e6:.2f}M")

vesselFM folder already present.



torch version: 2.10.0+cu128
CUDA available: True

vesselFM loaded on cuda:0. Total params: 31.42M


In [ ]:
import os
import subprocess
import sys

def pip(args):
    result = subprocess.run(["pip"] + args, capture_output=True, text=True)
    if result.returncode != 0:
        print("PIP STEP FAILED:", args)
        print(result.stderr[-1500:])
    return result.returncode

if not os.path.exists("/kaggle/working/vesselFM"):
    print("Cloning vesselFM...")
    ret = os.system("git clone --quiet https://github.com/bwittmann/vesselFM /kaggle/working/vesselFM")
    if ret != 0:
        raise RuntimeError("git clone failed -- check network access in notebook settings.")
else:
    print("vesselFM folder already present.")

config_path = "/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml"
assert os.path.exists(config_path), f"Config missing: {config_path}"

pip(["install", "-q", "monai", "hydra-core", "omegaconf", "huggingface_hub", "scikit-image", "nibabel", "tqdm"])
pip(["install", "-q", "-e", "/kaggle/working/vesselFM", "--no-deps"])

eval_file = "/kaggle/working/vesselFM/vesselfm/seg/utils/evaluation.py"
with open(eval_file, "r") as f:
    content = f.read()
if "skeletonize_3d = skeletonize" not in content:
    content = content.replace(
        "from skimage.morphology import skeletonize, skeletonize_3d",
        "from skimage.morphology import skeletonize\nskeletonize_3d = skeletonize"
    )
    with open(eval_file, "w") as f:
        f.write(content)
    print("Patched evaluation.py for skeletonize_3d compatibility.")
else:
    print("evaluation.py already patched.")

import numpy
print("\nnumpy version (untouched, should be Kaggle default, NOT 1.26.4):", numpy.__version__)

import torch
print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

sys.path.insert(0, "/kaggle/working/vesselFM")
import hydra
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model_cfg = OmegaConf.load(config_path)
ckpt_path = hf_hub_download(repo_id="bwittmann/vesselFM", filename="vesselFM_base.pt")
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)

model = hydra.utils.instantiate(model_cfg)
model.load_state_dict(ckpt)
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"\nvesselFM loaded on {device}. Total params: {n_params/1e6:.2f}M")

In [ ]:
import subprocess

result = subprocess.run(['pip', 'install', '-q', '--dry-run', 'monai', 'hydra-core', 'omegaconf',
                          'huggingface_hub', 'scikit-image', 'nibabel', 'tqdm'],
                         capture_output=True, text=True)
print('STDOUT:')
print(result.stdout)
print('STDERR:')
print(result.stderr[-2000:])

In [ ]:
import subprocess
import os

def run(cmd):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('FAILED, stderr:')
        print(result.stderr[-2000:])
    else:
        print('OK')
    return result.returncode

run('python -m venv /kaggle/working/venv --system-site-packages')

pip_bin = '/kaggle/working/venv/bin/pip'

run(f'{pip_bin} install -q --upgrade pip')
run(f'{pip_bin} install -q monai hydra-core omegaconf huggingface_hub scikit-image nibabel tqdm')

print('\nVenv created and packages installed.')
print('numpy inside venv:')
run(f'{pip_bin} show numpy | grep Version')

In [ ]:
import subprocess

result = subprocess.run(['pip', 'list', '-v'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if 'numpy' in line.lower():
        print(line)

print('\n--- find command ---')
result2 = subprocess.run(['find', '/', '-maxdepth', '6', '-iname', 'numpy', '-type', 'd'],
                          capture_output=True, text=True)
print(result2.stdout)

print('\n--- python sys.path ---')
import sys
for p in sys.path:
    print(p)

In [ ]:
import subprocess
import os

def pip(args):
    result = subprocess.run(['pip'] + args, capture_output=True, text=True)
    if result.returncode != 0:
        print('PIP STEP FAILED:', args)
        print(result.stderr[-1000:])
    return result.returncode

import numpy
print('numpy version BEFORE any install:', numpy.__version__)

if not os.path.exists('/kaggle/working/vesselFM'):
    os.system('git clone --quiet https://github.com/bwittmann/vesselFM /kaggle/working/vesselFM')

config_path = '/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml'
assert os.path.exists(config_path)

eval_file = '/kaggle/working/vesselFM/vesselfm/seg/utils/evaluation.py'
with open(eval_file, 'r') as f:
    content = f.read()
if 'skeletonize_3d = skeletonize' not in content:
    content = content.replace(
        'from skimage.morphology import skeletonize, skeletonize_3d',
        'from skimage.morphology import skeletonize\nskeletonize_3d = skeletonize'
    )
    with open(eval_file, 'w') as f:
        f.write(content)

pip(['install', '-q', '--no-deps', 'monai'])
pip(['install', '-q', '--no-deps', 'scikit-image'])
pip(['install', '-q', '--no-deps', 'hydra-core', 'omegaconf', 'huggingface_hub', 'nibabel', 'tqdm'])
pip(['install', '-q', '-e', '/kaggle/working/vesselFM', '--no-deps'])
pip(['install', '-q', '--no-deps', 'lazy_loader', 'imageio', 'tifffile', 'networkx',
     'pywavelets', 'itk', 'einops', 'fire', 'pyyaml', 'termcolor'])

import numpy
print('numpy version AFTER installs:', numpy.__version__)

import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

import sys
sys.path.insert(0, '/kaggle/working/vesselFM')
import hydra
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model_cfg = OmegaConf.load(config_path)
ckpt_path = hf_hub_download(repo_id='bwittmann/vesselFM', filename='vesselFM_base.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)

model = hydra.utils.instantiate(model_cfg)
model.load_state_dict(ckpt)
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'\nvesselFM loaded on {device}. Total params: {n_params/1e6:.2f}M')

In [ ]:
import subprocess

result = subprocess.run(
    ['pip', 'install', '-q', '--force-reinstall', '--no-deps', 'numpy'],
    capture_output=True, text=True
)
print(result.stdout[-1000:])
print(result.stderr[-1000:])

import importlib
import numpy
importlib.reload(numpy)
print('\nnumpy version after clean reinstall:', numpy.__version__)

In [ ]:
import numpy
print('numpy version:', numpy.__version__)

import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

import sys
sys.path.insert(0, '/kaggle/working/vesselFM')
import hydra
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

config_path = '/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml'
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model_cfg = OmegaConf.load(config_path)
ckpt_path = hf_hub_download(repo_id='bwittmann/vesselFM', filename='vesselFM_base.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)

model = hydra.utils.instantiate(model_cfg)
model.load_state_dict(ckpt)
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'\nvesselFM loaded on {device}. Total params: {n_params/1e6:.2f}M')

In [ ]:
import subprocess

# reinstall numpy BEFORE it gets imported anywhere in this fresh process --
# this avoids the reload() problem we hit last time
result = subprocess.run(
    ['pip', 'install', '-q', '--force-reinstall', '--no-deps', 'numpy'],
    capture_output=True, text=True
)
print('numpy reinstall stdout (last part):', result.stdout[-500:])
if result.returncode != 0:
    print('numpy reinstall FAILED:', result.stderr[-1000:])

# now import numpy for the first time in this process -- fresh files, no stale reload
import numpy
print('numpy version (first import, fresh process):', numpy.__version__)

import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

import sys
sys.path.insert(0, '/kaggle/working/vesselFM')
import hydra
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

config_path = '/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml'
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model_cfg = OmegaConf.load(config_path)
ckpt_path = hf_hub_download(repo_id='bwittmann/vesselFM', filename='vesselFM_base.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)

model = hydra.utils.instantiate(model_cfg)
model.load_state_dict(ckpt)
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'\nvesselFM loaded on {device}. Total params: {n_params/1e6:.2f}M')

In [1]:
import numpy
print('numpy version:', numpy.__version__)

try:
    import numpy.random
    print('numpy.random imported OK -- numpy itself is fine on its own')
except Exception as e:
    print('numpy.random FAILED on its own (confirms numpy itself is broken):')
    print(repr(e))

numpy version: 2.5.1
numpy.random imported OK -- numpy itself is fine on its own


In [2]:
import numpy
print('numpy version:', numpy.__version__)

try:
    import monai
    print('monai imported OK, version:', monai.__version__)
except Exception as e:
    print('monai import FAILED directly:')
    import traceback
    traceback.print_exc()

numpy version: 2.5.1
monai imported OK, version: 1.6.0


In [3]:
import numpy
print('numpy version:', numpy.__version__)

import monai
print('monai imported OK:', monai.__version__)

from monai.networks.nets import DynUNet
print('DynUNet imported directly (no hydra) OK')

import hydra
print('hydra imported OK, version:', hydra.__version__)

try:
    obj = hydra._internal.utils._locate('monai.networks.nets.DynUNet')
    print('hydra._locate succeeded:', obj)
except Exception as e:
    print('hydra._locate FAILED:')
    import traceback
    traceback.print_exc()

numpy version: 2.5.1
monai imported OK: 1.6.0
DynUNet imported directly (no hydra) OK
hydra imported OK, version: 1.3.4
hydra._locate succeeded: <class 'monai.networks.nets.dynunet.DynUNet'>


In [4]:
import numpy
print('numpy version:', numpy.__version__)

import torch
print('torch imported OK, version:', torch.__version__)

import sys
sys.path.insert(0, '/kaggle/working/vesselFM')

import monai
print('monai imported OK:', monai.__version__)

import hydra
from omegaconf import OmegaConf
print('hydra, omegaconf imported OK')

config_path = '/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml'
model_cfg = OmegaConf.load(config_path)
print('config loaded OK')
print(model_cfg)

model = hydra.utils.instantiate(model_cfg)
print('\nmodel instantiated OK:', type(model))

numpy version: 2.5.1
torch imported OK, version: 2.10.0+cu128
monai imported OK: 1.6.0
hydra, omegaconf imported OK
config loaded OK
{'_target_': 'monai.networks.nets.DynUNet', 'in_channels': 1, 'out_channels': 1, 'spatial_dims': 3, 'strides': [[1, 1, 1], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2]], 'kernel_size': [[3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3], [3, 3, 3]], 'upsample_kernel_size': [[2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2]], 'filters': [32, 64, 128, 256, 320, 320], 'res_block': True}

model instantiated OK: <class 'monai.networks.nets.dynunet.DynUNet'>


In [5]:
import numpy
print('numpy version:', numpy.__version__)

import torch
import sys
sys.path.insert(0, '/kaggle/working/vesselFM')

import monai
import hydra
from omegaconf import OmegaConf

config_path = '/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml'
model_cfg = OmegaConf.load(config_path)
model = hydra.utils.instantiate(model_cfg)
print('model instantiated OK')

print('\nnumpy version right before hf_hub_download:', numpy.__version__)

from huggingface_hub import hf_hub_download
ckpt_path = hf_hub_download(repo_id='bwittmann/vesselFM', filename='vesselFM_base.pt')
print('checkpoint downloaded OK:', ckpt_path)

print('numpy version right after hf_hub_download:', numpy.__version__)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
print('checkpoint loaded OK')

model.load_state_dict(ckpt)
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'\nvesselFM fully loaded on {device}. Total params: {n_params/1e6:.2f}M')

numpy version: 2.5.1
model instantiated OK

numpy version right before hf_hub_download: 2.5.1
checkpoint downloaded OK: /root/.cache/huggingface/hub/models--bwittmann--vesselFM/snapshots/ad90fa23318ecde5f54c2dfdc6ded0b6740b9bc9/vesselFM_base.pt
numpy version right after hf_hub_download: 2.5.1
checkpoint loaded OK

vesselFM fully loaded on cuda:0. Total params: 31.42M


In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib
from PIL import Image
from tqdm import tqdm
import os

jp2_dir = Path('/kaggle/input/datasets/rabeya100x/kidney-complete-organ222')

if not jp2_dir.exists():
    print('Still not found:', jp2_dir)
    parent = Path('/kaggle/input/datasets/rabeya100x')
    print('Contents of parent folder instead:')
    for item in parent.iterdir():
        print(' -', item.name)
    raise FileNotFoundError('Fix jp2_dir path above, then re-run this cell.')

slice_files = sorted(jp2_dir.glob('*.jp2'))
print('Total slices found:', len(slice_files))

if len(slice_files) == 0:
    print('No jp2 files directly inside, checking one level deeper...')
    for item in jp2_dir.iterdir():
        print(' -', item.name)
    raise FileNotFoundError('No jp2 files found at this level, check the listing above.')

first_slice = np.array(Image.open(slice_files[0]))
print('Single slice shape:', first_slice.shape, 'dtype:', first_slice.dtype)

os.makedirs('/kaggle/working/real_input', exist_ok=True)
nii_path = '/kaggle/working/real_input/kidney_K292_163um.nii.gz'

if os.path.exists(nii_path):
    print('nii.gz already exists, skipping conversion.')
else:
    h, w = first_slice.shape
    d = len(slice_files)
    print('Volume shape will be:', (h, w, d))

    volume = np.zeros((h, w, d), dtype=np.uint16)
    for i, f in enumerate(tqdm(slice_files, desc='Loading slices')):
        volume[:, :, i] = np.array(Image.open(f))

    nifti_img = nib.Nifti1Image(volume.astype(np.float32), affine=np.eye(4))
    nib.save(nifti_img, nii_path)
    print('Saved:', nii_path)

print('File size MB:', os.path.getsize(nii_path) / 1e6)

In [ ]:
eval_file = '/kaggle/working/vesselFM/vesselfm/seg/utils/evaluation.py'

with open(eval_file, 'r') as f:
    content = f.read()

original_content = content

content = content.replace(
    'from skimage.morphology import skeletonize, skeletonize_3d',
    'from skimage.morphology import skeletonize\nskeletonize_3d = skeletonize'
)

if content == original_content:
    print('No change made -- the import line may look different than expected.')
    print('Searching for any line containing skeletonize_3d:')
    for i, line in enumerate(original_content.splitlines()):
        if 'skeletonize_3d' in line:
            print(i, ':', line)
else:
    with open(eval_file, 'w') as f:
        f.write(content)
    print('Patched successfully:', eval_file)

with open(eval_file, 'r') as f:
    for i, line in enumerate(f.read().splitlines()[:10]):
        print(i, line)

In [ ]:
import os

os.makedirs('/kaggle/working/real_output', exist_ok=True)

pred_path = '/kaggle/working/real_output/kidney_K292_163um_pred.nii.gz'

if os.path.exists(pred_path):
    print('Prediction already exists, skipping inference.')
else:
    cmd = (
        'cd /kaggle/working/vesselFM && python -m vesselfm.seg.inference '
        'image_path=/kaggle/working/real_input '
        'output_folder=/kaggle/working/real_output '
        'image_file_ending=nii.gz device=cuda:0 batch_size=1 overlap=0.25'
    )
    print('Running:', cmd)
    exit_code = os.system(cmd)
    print('Exit code:', exit_code)

print('Contents of real_output:')
for f in os.listdir('/kaggle/working/real_output'):
    print(' -', f)

In [ ]:
import nibabel as nib
import numpy as np

orig = nib.load('/kaggle/working/real_input/kidney_K292_163um.nii.gz').get_fdata()
pred = nib.load('/kaggle/working/real_output/kidney_K292_163um_pred.nii.gz').get_fdata()

print('orig shape:', orig.shape)
print('pred shape:', pred.shape)
print('orig dtype:', orig.dtype, '| pred dtype:', pred.dtype)
print('orig value range:', orig.min(), '-', orig.max())
print('pred value range:', pred.min(), '-', pred.max())

vessel_voxels = np.sum(pred > 0.5)
total_voxels = pred.size
print('predicted vessel voxels (>0.5):', vessel_voxels)
print('fraction of volume predicted as vessel:', vessel_voxels / total_voxels)

if orig.shape != pred.shape:
    print('WARNING: orig and pred shapes do not match, something is misaligned.')
else:
    print('Shapes match, orig and pred are spatially aligned.')

In [ ]:
import numpy as np
from skimage.measure import regionprops, label
from skimage.filters import sato
from scipy import ndimage


def compute_collapse_score(img_slice, percentile_threshold=97, min_area=150):
    threshold = np.percentile(img_slice, percentile_threshold)
    mask = img_slice > threshold
    labeled_mask = label(mask)
    regions = regionprops(labeled_mask)

    dist_transform = ndimage.distance_transform_edt(mask)
    vesselness_map = sato(img_slice, sigmas=range(1, 6), black_ridges=False)

    scores = []
    for r in regions:
        if r.area <= min_area:
            continue

        region_mask = labeled_mask == r.label

        eccentricity = r.eccentricity

        local_thickness = dist_transform[region_mask]
        if local_thickness.size > 0:
            thickness_ratio = local_thickness.std() / (local_thickness.mean() + 1e-8)
        else:
            thickness_ratio = np.nan

        major = r.major_axis_length
        minor = r.minor_axis_length if r.minor_axis_length > 0 else 1e-8
        hessian_ratio = major / minor

        vesselness_val = vesselness_map[region_mask].mean()

        norm_eccentricity = eccentricity
        norm_thickness = min(thickness_ratio / 1.0, 1.0) if not np.isnan(thickness_ratio) else np.nan
        norm_hessian = min(hessian_ratio / 10.0, 1.0)
        norm_vesselness = 1.0 - min(vesselness_val / (vesselness_map.max() + 1e-8), 1.0)

        collapse_score = np.nanmean([norm_eccentricity, norm_thickness, norm_hessian, norm_vesselness])

        scores.append({
            'label': r.label,
            'area': r.area,
            'eccentricity': eccentricity,
            'thickness_ratio': thickness_ratio,
            'hessian_ratio': hessian_ratio,
            'vesselness': vesselness_val,
            'collapse_score': collapse_score,
        })

    return scores, labeled_mask


test_scores, test_labeled = compute_collapse_score(orig[:, :, mid_slice_idx])

print('Regions kept:', len(test_scores))
for s in sorted(test_scores, key=lambda x: -x['area'])[:15]:
    print(f"  area={s['area']:6.0f}  ecc={s['eccentricity']:.3f}  "
          f"thick_ratio={s['thickness_ratio']:.3f}  hess_ratio={s['hessian_ratio']:.3f}  "
          f"vesselness={s['vesselness']:.4f}  collapse_score={s['collapse_score']:.3f}")

collapse_vals = [s['collapse_score'] for s in test_scores if not np.isnan(s['collapse_score'])]
if collapse_vals:
    print('\ncollapse_score range:', min(collapse_vals), '-', max(collapse_vals))
    print('collapse_score mean:', np.mean(collapse_vals))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PhysicsAwareAdapter(nn.Module):
    """
    Lightweight residual adapter. Takes a backbone feature map plus a
    per-voxel collapse_score map (from compute_collapse_score, precomputed
    offline) and learns a correction so the network stops penalizing
    flattened/elliptical vessel shapes in high-collapse regions.

    Base vesselFM weights stay frozen; only this adapter is trained
    (LoRA-style, cheap to fine-tune on few-shot data).
    """

    def __init__(self, in_channels, bottleneck_channels=16):
        super().__init__()
        self.down = nn.Conv3d(in_channels + 1, bottleneck_channels, kernel_size=1)
        self.act = nn.GELU()
        self.spatial = nn.Conv3d(
            bottleneck_channels, bottleneck_channels,
            kernel_size=3, padding=1, groups=bottleneck_channels
        )
        self.up = nn.Conv3d(bottleneck_channels, in_channels, kernel_size=1)

        # zero-init so the adapter starts as an identity function,
        # meaning it cannot break the pretrained backbone at step 0
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, feat, collapse_score_map):
        collapse_score_map = F.interpolate(
            collapse_score_map, size=feat.shape[2:], mode='trilinear', align_corners=False
        )
        x = torch.cat([feat, collapse_score_map], dim=1)
        x = self.act(self.down(x))
        x = self.act(self.spatial(x))
        delta = self.up(x)
        return feat + delta


class CollapseAwareLoss(nn.Module):
    """
    Standard segmentation loss (Dice + BCE) plus a shape-prior term that
    relaxes the implicit "must be round" bias in regions with high
    collapse_score, so the network is not punished for correctly predicting
    an elongated, flattened vessel where the physics says it should be.
    """

    def __init__(self, dice_weight=1.0, bce_weight=1.0, shape_weight=0.5, smooth=1e-5):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.shape_weight = shape_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def dice_loss(self, probs, target):
        dims = (2, 3, 4)
        intersection = (probs * target).sum(dim=dims)
        union = probs.sum(dim=dims) + target.sum(dim=dims)
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()

    def shape_penalty(self, probs, collapse_score_map):
        dz = torch.abs(probs[:, :, 1:, :, :] - probs[:, :, :-1, :, :])
        dy = torch.abs(probs[:, :, :, 1:, :] - probs[:, :, :, :-1, :])
        dx = torch.abs(probs[:, :, :, :, 1:] - probs[:, :, :, :, :-1])

        weight = 1.0 - collapse_score_map

        tv = (
            (dz * weight[:, :, 1:, :, :]).mean()
            + (dy * weight[:, :, :, 1:, :]).mean()
            + (dx * weight[:, :, :, :, 1:]).mean()
        )
        return tv

    def forward(self, logits, target, collapse_score_map):
        probs = torch.sigmoid(logits)

        bce = self.bce(logits, target).mean()
        dice = self.dice_loss(probs, target)
        shape = self.shape_penalty(probs, collapse_score_map)

        total = self.bce_weight * bce + self.dice_weight * dice + self.shape_weight * shape
        return total, {'bce': bce.item(), 'dice': dice.item(), 'shape': shape.item()}


device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

adapter = PhysicsAwareAdapter(in_channels=32).to(device)
dummy_feat = torch.randn(1, 32, 16, 16, 16).to(device)
dummy_score = torch.rand(1, 1, 128, 128, 128).to(device)
out = adapter(dummy_feat, dummy_score)
print('Adapter output shape:', out.shape)
print('Identity at init (should be True):', torch.allclose(out, dummy_feat, atol=1e-5))

criterion = CollapseAwareLoss().to(device)
logits = torch.randn(1, 1, 32, 32, 32, requires_grad=True, device=device)
target = (torch.rand(1, 1, 32, 32, 32, device=device) > 0.7).float()
collapse_map = torch.rand(1, 1, 32, 32, 32, device=device)

loss, parts = criterion(logits, target, collapse_map)
loss.backward()
print('\nLoss total:', loss.item())
print('Loss breakdown:', parts)
print('Gradients flowing:', logits.grad is not None)

In [ ]:
import torch

print('=== Conv3d / ConvTranspose3d layers in the model ===')
for name, module in model.named_modules():
    if isinstance(module, (torch.nn.Conv3d, torch.nn.ConvTranspose3d)):
        n_params = sum(p.numel() for p in module.parameters())
        print(f'{name:55s} {module.__class__.__name__:18s} '
              f'in={getattr(module, "in_channels", "?")} out={getattr(module, "out_channels", "?")} '
              f'params={n_params}')

print('\n=== Top-level model structure (class names only, 2 levels deep) ===')
for name, module in model.named_children():
    print(name, ':', module.__class__.__name__)
    for sub_name, sub_module in module.named_children():
        print('   ', sub_name, ':', sub_module.__class__.__name__)

In [ ]:
import torch

collapse_map_holder = {'map': None}

adapter = PhysicsAwareAdapter(in_channels=32).to(device)

def adapter_pre_hook(module, args):
    feat = args[0]
    collapse_map = collapse_map_holder['map']
    if collapse_map is None:
        return None
    adjusted_feat = adapter(feat, collapse_map)
    return (adjusted_feat,)

output_conv_module = dict(model.named_modules())['output_block.conv.conv']
hook_handle = output_conv_module.register_forward_pre_hook(adapter_pre_hook)

print('Hook registered on: output_block.conv.conv')
print('Adapter param count:', sum(p.numel() for p in adapter.parameters()))

dummy_input = torch.randn(1, 1, 32, 128, 128).to(device)
dummy_collapse_map = torch.zeros(1, 1, 32, 128, 128).to(device)

model.eval()
with torch.no_grad():
    collapse_map_holder['map'] = None
    out_no_adapter = model(dummy_input)

    collapse_map_holder['map'] = dummy_collapse_map
    out_with_adapter = model(dummy_input)

print('\nOutput shape:', out_with_adapter.shape)
print('Adapter untrained + zero collapse_score -> output should match baseline')
print('Outputs identical:', torch.allclose(out_no_adapter, out_with_adapter, atol=1e-5))

hook_handle.remove()
print('\nHook removed for this test. Will re-register right before real training/inference.')

Part 9 — real SenNet+HOA dataset (kidney_1_dense)

In [ ]:
import os

base = '/kaggle/input/competitions/blood-vessel-segmentation'
print('Top level of competition folder:')
for item in sorted(os.listdir(base)):
    print(' -', item)

train_dir = os.path.join(base, 'train')
if os.path.exists(train_dir):
    print('\nContents of train/:')
    for item in sorted(os.listdir(train_dir)):
        print(' -', item)

kidney1_dir = os.path.join(train_dir, 'kidney_1_dense')
if os.path.exists(kidney1_dir):
    print('\nContents of train/kidney_1_dense/:')
    for item in sorted(os.listdir(kidney1_dir)):
        full = os.path.join(kidney1_dir, item)
        if os.path.isdir(full):
            n_files = len(os.listdir(full))
            print(f' - {item}/  ({n_files} files)')
        else:
            print(' -', item)
else:
    print('\nkidney_1_dense not found at expected path, listing train/ contents above to find correct name.')

Part 9 — Dataset class (vessel-biased sampling directly, skipping the basic non-biased version to save time)

In [ ]:
import numpy as np
from PIL import Image

sample_indices = list(range(0, train_dataset.n_slices, 50))
vessel_fraction_per_slice = []
img_stats = []

for idx in sample_indices:
    img = np.array(Image.open(train_dataset.image_files[idx]))
    lbl = np.array(Image.open(train_dataset.label_files[idx]))
    vessel_frac = (lbl > 0).mean()
    vessel_fraction_per_slice.append(vessel_frac)
    img_stats.append((img.min(), img.max(), img.mean(), img.dtype))

vessel_fraction_per_slice = np.array(vessel_fraction_per_slice)
print('Checked', len(sample_indices), 'slices out of', train_dataset.n_slices)
print('Slices with any vessel pixel:', (vessel_fraction_per_slice > 0).sum(), '/', len(sample_indices))
print('Mean vessel fraction (over slices that have any):',
      vessel_fraction_per_slice[vessel_fraction_per_slice > 0].mean()
      if (vessel_fraction_per_slice > 0).any() else 'none found')
print('Max vessel fraction seen:', vessel_fraction_per_slice.max())

print('\nImage dtype (first sample):', img_stats[0][3])
print('Image min/max/mean across sampled slices:')
for i, idx in enumerate(sample_indices[:10]):
    print(f'  slice {idx:5d}  min={img_stats[i][0]:8.1f}  max={img_stats[i][1]:8.1f}  mean={img_stats[i][2]:8.1f}')

best_idx = sample_indices[np.argmax(vessel_fraction_per_slice)]
print('\nBest vessel-density slice found:', best_idx, 'fraction:', vessel_fraction_per_slice.max())

test_img = np.array(Image.open(train_dataset.image_files[best_idx])).astype(np.float64)
print('\nPercentile check on this slice for compute_collapse_score tuning:')
for pct in [85, 90, 95, 97, 99]:
    thresh_val = np.percentile(test_img, pct)
    n_above = (test_img > thresh_val).sum()
    print(f'  percentile={pct}  thresh_val={thresh_val:.2f}  pixels_above={n_above}')

In [ ]:
import numpy as np
from PIL import Image
from scipy import ndimage

best_idx = 950

img = np.array(Image.open(train_dataset.image_files[best_idx])).astype(np.float64)
lbl = np.array(Image.open(train_dataset.label_files[best_idx]))
lbl_binary = (lbl > 0).astype(np.uint8)

print('Image shape:', img.shape)
print('Label shape:', lbl.shape)
print('Vessel pixels in this slice:', lbl_binary.sum())

ys, xs = np.where(lbl_binary > 0)
print('Vessel pixel y-range:', ys.min(), '-', ys.max())
print('Vessel pixel x-range:', xs.min(), '-', xs.max())
print('Vessel centroid (y, x):', ys.mean(), xs.mean())

scores, labeled_mask = compute_collapse_score(img, percentile_threshold=97, min_area=150)
print('\nRegions found by compute_collapse_score on this real slice:', len(scores))
if scores:
    for s in scores[:5]:
        print('  area=', s['area'], 'collapse_score=', round(s['collapse_score'], 3))

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from pathlib import Path


class KidneyVesselDataset(Dataset):
    def __init__(self, kidney_dir, patch_size=(16, 128, 128),
                 percentile_threshold=97, min_area=150, vessel_bias_prob=0.9):
        self.kidney_dir = Path(kidney_dir)
        self.image_dir = self.kidney_dir / 'images'
        self.label_dir = self.kidney_dir / 'labels'

        self.image_files = sorted(self.image_dir.glob('*.tif'))
        self.label_files = sorted(self.label_dir.glob('*.tif'))
        assert len(self.image_files) == len(self.label_files), 'image/label count mismatch'

        self.patch_depth, self.patch_h, self.patch_w = patch_size
        self.percentile_threshold = percentile_threshold
        self.min_area = min_area
        self.vessel_bias_prob = vessel_bias_prob
        self.n_slices = len(self.image_files)

    def __len__(self):
        return max(self.n_slices - self.patch_depth, 1)

    def _load_slice(self, idx):
        img = np.array(Image.open(self.image_files[idx])).astype(np.float32)
        lbl = np.array(Image.open(self.label_files[idx])).astype(np.float32)
        return img, (lbl > 0).astype(np.float32)

    def _find_vessel_anchor(self):
        for _ in range(15):
            idx = np.random.randint(0, len(self))
            lbl = np.array(Image.open(self.label_files[idx]))
            ys, xs = np.where(lbl > 0)
            if len(ys) > 0:
                cy = int(ys.mean())
                cx = int(xs.mean())
                return idx, cy, cx
        return np.random.randint(0, len(self)), None, None

    def __getitem__(self, idx):
        if np.random.rand() < self.vessel_bias_prob:
            z0, anchor_y, anchor_x = self._find_vessel_anchor()
        else:
            z0, anchor_y, anchor_x = np.random.randint(0, len(self)), None, None

        z1 = z0 + self.patch_depth
        if z1 > self.n_slices:
            z0 = self.n_slices - self.patch_depth
            z1 = self.n_slices

        first_img, _ = self._load_slice(z0)
        H, W = first_img.shape
        ph, pw = min(self.patch_h, H), min(self.patch_w, W)

        if anchor_y is not None:
            jitter_y = np.random.randint(-ph // 4, ph // 4 + 1)
            jitter_x = np.random.randint(-pw // 4, pw // 4 + 1)
            y0 = np.clip(anchor_y - ph // 2 + jitter_y, 0, H - ph)
            x0 = np.clip(anchor_x - pw // 2 + jitter_x, 0, W - pw)
        else:
            y0 = np.random.randint(0, H - ph + 1)
            x0 = np.random.randint(0, W - pw + 1)

        images, labels, collapse_scores = [], [], []
        for z in range(z0, z1):
            img, lbl = self._load_slice(z)
            img_crop = img[y0:y0 + ph, x0:x0 + pw]
            lbl_crop = lbl[y0:y0 + ph, x0:x0 + pw]

            scores, labeled_mask = compute_collapse_score(
                img, percentile_threshold=self.percentile_threshold, min_area=self.min_area
            )
            score_slice = np.zeros_like(img, dtype=np.float32)
            for s in scores:
                if np.isnan(s['collapse_score']):
                    continue
                score_slice[labeled_mask == s['label']] = s['collapse_score']
            score_crop = score_slice[y0:y0 + ph, x0:x0 + pw]

            images.append(img_crop)
            labels.append(lbl_crop)
            collapse_scores.append(score_crop)

        image_vol = np.stack(images, axis=0)
        label_vol = np.stack(labels, axis=0)
        collapse_vol = np.stack(collapse_scores, axis=0)

        image_vol = (image_vol - image_vol.mean()) / (image_vol.std() + 1e-8)

        return {
            'image': torch.from_numpy(image_vol).unsqueeze(0).float(),
            'mask': torch.from_numpy(label_vol).unsqueeze(0).float(),
            'collapse_score': torch.from_numpy(collapse_vol).unsqueeze(0).float(),
        }


train_dataset = KidneyVesselDataset(
    '/kaggle/input/competitions/blood-vessel-segmentation/train/kidney_1_dense',
    patch_size=(16, 128, 128),
)
print('Dataset length:', len(train_dataset))

vessel_counts = []
collapse_maxes = []
for i in range(10):
    sample = train_dataset[i]
    vessel_counts.append(sample['mask'].sum().item())
    collapse_maxes.append(sample['collapse_score'].max().item())

print('Vessel voxel counts across 10 samples:', vessel_counts)
print('Fraction with vessels present:', sum(v > 0 for v in vessel_counts) / len(vessel_counts))
print('Collapse score max across 10 samples:', collapse_maxes)

In [ ]:
print('strides:', model.strides)
print('kernel_size:', model.kernel_size)
print('upsample_kernel_size:', model.upsample_kernel_size)
print('deep_supervision:', model.deep_supervision if hasattr(model, 'deep_supervision') else 'N/A')

In [ ]:
train_dataset = KidneyVesselDataset(
    '/kaggle/input/competitions/blood-vessel-segmentation/train/kidney_1_dense',
    patch_size=(32, 128, 128),
)
print('Dataset length:', len(train_dataset))

sample = train_dataset[0]
print('image shape:', sample['image'].shape)
print('mask shape:', sample['mask'].shape, '| vessel voxels:', sample['mask'].sum().item())
print('collapse_score shape:', sample['collapse_score'].shape, '| max:', sample['collapse_score'].max().item())

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from pathlib import Path


class KidneyVesselDataset(Dataset):
    def __init__(self, kidney_dir, patch_size=(32, 128, 128),
                 percentile_threshold=97, min_area=150, vessel_bias_prob=0.9):
        self.kidney_dir = Path(kidney_dir)
        self.image_dir = self.kidney_dir / 'images'
        self.label_dir = self.kidney_dir / 'labels'

        self.image_files = sorted(self.image_dir.glob('*.tif'))
        self.label_files = sorted(self.label_dir.glob('*.tif'))
        assert len(self.image_files) == len(self.label_files), 'image/label count mismatch'

        self.patch_depth, self.patch_h, self.patch_w = patch_size
        self.percentile_threshold = percentile_threshold
        self.min_area = min_area
        self.vessel_bias_prob = vessel_bias_prob
        self.n_slices = len(self.image_files)

        self._collapse_cache = {}

    def __len__(self):
        return max(self.n_slices - self.patch_depth, 1)

    def _load_slice(self, idx):
        img = np.array(Image.open(self.image_files[idx])).astype(np.float32)
        lbl = np.array(Image.open(self.label_files[idx])).astype(np.float32)
        return img, (lbl > 0).astype(np.float32)

    def _get_collapse_score(self, idx, img):
        if idx in self._collapse_cache:
            return self._collapse_cache[idx]

        scores, labeled_mask = compute_collapse_score(
            img, percentile_threshold=self.percentile_threshold, min_area=self.min_area
        )
        score_slice = np.zeros_like(img, dtype=np.float32)
        for s in scores:
            if np.isnan(s['collapse_score']):
                continue
            score_slice[labeled_mask == s['label']] = s['collapse_score']

        self._collapse_cache[idx] = score_slice
        return score_slice

    def _find_vessel_anchor(self):
        for _ in range(15):
            idx = np.random.randint(0, len(self))
            lbl = np.array(Image.open(self.label_files[idx]))
            ys, xs = np.where(lbl > 0)
            if len(ys) > 0:
                cy = int(ys.mean())
                cx = int(xs.mean())
                return idx, cy, cx
        return np.random.randint(0, len(self)), None, None

    def __getitem__(self, idx):
        if np.random.rand() < self.vessel_bias_prob:
            z0, anchor_y, anchor_x = self._find_vessel_anchor()
        else:
            z0, anchor_y, anchor_x = np.random.randint(0, len(self)), None, None

        z1 = z0 + self.patch_depth
        if z1 > self.n_slices:
            z0 = self.n_slices - self.patch_depth
            z1 = self.n_slices

        first_img, _ = self._load_slice(z0)
        H, W = first_img.shape
        ph, pw = min(self.patch_h, H), min(self.patch_w, W)

        if anchor_y is not None:
            jitter_y = np.random.randint(-ph // 4, ph // 4 + 1)
            jitter_x = np.random.randint(-pw // 4, pw // 4 + 1)
            y0 = np.clip(anchor_y - ph // 2 + jitter_y, 0, H - ph)
            x0 = np.clip(anchor_x - pw // 2 + jitter_x, 0, W - pw)
        else:
            y0 = np.random.randint(0, H - ph + 1)
            x0 = np.random.randint(0, W - pw + 1)

        images, labels, collapse_scores = [], [], []
        for z in range(z0, z1):
            img, lbl = self._load_slice(z)
            score_slice = self._get_collapse_score(z, img)

            img_crop = img[y0:y0 + ph, x0:x0 + pw]
            lbl_crop = lbl[y0:y0 + ph, x0:x0 + pw]
            score_crop = score_slice[y0:y0 + ph, x0:x0 + pw]

            images.append(img_crop)
            labels.append(lbl_crop)
            collapse_scores.append(score_crop)

        image_vol = np.stack(images, axis=0)
        label_vol = np.stack(labels, axis=0)
        collapse_vol = np.stack(collapse_scores, axis=0)

        image_vol = (image_vol - image_vol.mean()) / (image_vol.std() + 1e-8)

        return {
            'image': torch.from_numpy(image_vol).unsqueeze(0).float(),
            'mask': torch.from_numpy(label_vol).unsqueeze(0).float(),
            'collapse_score': torch.from_numpy(collapse_vol).unsqueeze(0).float(),
        }


train_dataset = KidneyVesselDataset(
    '/kaggle/input/competitions/blood-vessel-segmentation/train/kidney_1_dense',
    patch_size=(32, 128, 128),
)
print('Dataset re-created with caching. Length:', len(train_dataset))


In [ ]:
import time
import numpy as np
from skimage.filters import sato

test_idx = 950
img = np.array(Image.open(train_dataset.image_files[test_idx])).astype(np.float32)

lbl = np.array(Image.open(train_dataset.label_files[test_idx]))
ys, xs = np.where(lbl > 0)
cy, cx = int(ys.mean()), int(xs.mean())

ph, pw = 128, 128
H, W = img.shape
y0 = np.clip(cy - ph // 2, 0, H - ph)
x0 = np.clip(cx - pw // 2, 0, W - pw)
crop = img[y0:y0 + ph, x0:x0 + pw]

print('Crop shape:', crop.shape)

t0 = time.time()
vesselness_crop = sato(crop, sigmas=range(1, 6), black_ridges=False)
t1 = time.time()
print('Sato on 128x128 crop (full 5 sigmas):', round(t1 - t0, 4), 's')

full_vesselness = sato(img, sigmas=range(1, 6), black_ridges=False)
full_crop_region = full_vesselness[y0:y0 + ph, x0:x0 + pw]

diff = np.abs(vesselness_crop - full_crop_region)
corr = np.corrcoef(vesselness_crop.ravel(), full_crop_region.ravel())[0, 1]
print('Crop-first vs full-then-crop -- mean_abs_diff:', round(diff.mean(), 5), ' correlation:', round(corr, 4))

est_per_slice_new = t1 - t0
print(f'\nEstimated time per training step (32 slices): {est_per_slice_new * 32:.2f}s')
print(f'Estimated total for 5 epochs x 30 steps: {est_per_slice_new * 32 * 150 / 60:.1f} minutes')

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from pathlib import Path


class KidneyVesselDataset(Dataset):
    def __init__(self, kidney_dir, patch_size=(32, 128, 128),
                 percentile_threshold=97, min_area=150, vessel_bias_prob=0.9):
        self.kidney_dir = Path(kidney_dir)
        self.image_dir = self.kidney_dir / 'images'
        self.label_dir = self.kidney_dir / 'labels'

        self.image_files = sorted(self.image_dir.glob('*.tif'))
        self.label_files = sorted(self.label_dir.glob('*.tif'))
        assert len(self.image_files) == len(self.label_files), 'image/label count mismatch'

        self.patch_depth, self.patch_h, self.patch_w = patch_size
        self.percentile_threshold = percentile_threshold
        self.min_area = min_area
        self.vessel_bias_prob = vessel_bias_prob
        self.n_slices = len(self.image_files)

    def __len__(self):
        return max(self.n_slices - self.patch_depth, 1)

    def _load_slice(self, idx):
        img = np.array(Image.open(self.image_files[idx])).astype(np.float32)
        lbl = np.array(Image.open(self.label_files[idx])).astype(np.float32)
        return img, (lbl > 0).astype(np.float32)

    def _find_vessel_anchor(self):
        for _ in range(15):
            idx = np.random.randint(0, len(self))
            lbl = np.array(Image.open(self.label_files[idx]))
            ys, xs = np.where(lbl > 0)
            if len(ys) > 0:
                cy = int(ys.mean())
                cx = int(xs.mean())
                return idx, cy, cx
        return np.random.randint(0, len(self)), None, None

    def _collapse_score_on_crop(self, img_crop):
        # runs compute_collapse_score directly on the small crop,
        # instead of the full slice, since only the crop region is
        # ever used -- this is what made training fast (Sato filter
        # cost scales with pixel count, and the crop is ~1% of the
        # full slice)
        scores, labeled_mask = compute_collapse_score(
            img_crop, percentile_threshold=self.percentile_threshold, min_area=self.min_area
        )
        score_crop = np.zeros_like(img_crop, dtype=np.float32)
        for s in scores:
            if np.isnan(s['collapse_score']):
                continue
            score_crop[labeled_mask == s['label']] = s['collapse_score']
        return score_crop

    def __getitem__(self, idx):
        if np.random.rand() < self.vessel_bias_prob:
            z0, anchor_y, anchor_x = self._find_vessel_anchor()
        else:
            z0, anchor_y, anchor_x = np.random.randint(0, len(self)), None, None

        z1 = z0 + self.patch_depth
        if z1 > self.n_slices:
            z0 = self.n_slices - self.patch_depth
            z1 = self.n_slices

        first_img, _ = self._load_slice(z0)
        H, W = first_img.shape
        ph, pw = min(self.patch_h, H), min(self.patch_w, W)

        if anchor_y is not None:
            jitter_y = np.random.randint(-ph // 4, ph // 4 + 1)
            jitter_x = np.random.randint(-pw // 4, pw // 4 + 1)
            y0 = np.clip(anchor_y - ph // 2 + jitter_y, 0, H - ph)
            x0 = np.clip(anchor_x - pw // 2 + jitter_x, 0, W - pw)
        else:
            y0 = np.random.randint(0, H - ph + 1)
            x0 = np.random.randint(0, W - pw + 1)

        images, labels, collapse_scores = [], [], []
        for z in range(z0, z1):
            img, lbl = self._load_slice(z)
            img_crop = img[y0:y0 + ph, x0:x0 + pw]
            lbl_crop = lbl[y0:y0 + ph, x0:x0 + pw]
            score_crop = self._collapse_score_on_crop(img_crop)

            images.append(img_crop)
            labels.append(lbl_crop)
            collapse_scores.append(score_crop)

        image_vol = np.stack(images, axis=0)
        label_vol = np.stack(labels, axis=0)
        collapse_vol = np.stack(collapse_scores, axis=0)

        image_vol = (image_vol - image_vol.mean()) / (image_vol.std() + 1e-8)

        return {
            'image': torch.from_numpy(image_vol).unsqueeze(0).float(),
            'mask': torch.from_numpy(label_vol).unsqueeze(0).float(),
            'collapse_score': torch.from_numpy(collapse_vol).unsqueeze(0).float(),
        }


train_dataset = KidneyVesselDataset(
    '/kaggle/input/competitions/blood-vessel-segmentation/train/kidney_1_dense',
    patch_size=(32, 128, 128),
)
print('Dataset re-created (crop-first, fast). Length:', len(train_dataset))

import time
t0 = time.time()
sample = train_dataset[5]
t1 = time.time()
print('Single __getitem__ took:', round(t1 - t0, 2), 's')
print('image:', sample['image'].shape)
print('mask:', sample['mask'].shape, '| vessel voxels:', sample['mask'].sum().item())
print('collapse_score:', sample['collapse_score'].shape, '| max:', sample['collapse_score'].max().item())

In [ ]:
import torch
from torch.utils.data import DataLoader
import time

torch.manual_seed(42)
np.random.seed(42)

model.eval()
for p in model.parameters():
    p.requires_grad = False

adapter = PhysicsAwareAdapter(in_channels=32).to(device)
criterion = CollapseAwareLoss(dice_weight=1.0, bce_weight=1.0, shape_weight=0.5).to(device)
optimizer = torch.optim.Adam(adapter.parameters(), lr=1e-3)

collapse_map_holder = {'map': None}

def adapter_pre_hook(module, args):
    feat = args[0]
    collapse_map = collapse_map_holder['map']
    if collapse_map is None:
        return None
    adjusted_feat = adapter(feat, collapse_map)
    return (adjusted_feat,)

output_conv_module = dict(model.named_modules())['output_block.conv.conv']
hook_handle = output_conv_module.register_forward_pre_hook(adapter_pre_hook)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)

n_epochs = 5
max_steps_per_epoch = 30

adapter.train()
history = []

for epoch in range(n_epochs):
    epoch_losses = []
    epoch_dices = []
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        if step >= max_steps_per_epoch:
            break

        image = batch['image'].to(device)
        mask = batch['mask'].to(device)
        collapse_score = batch['collapse_score'].to(device)

        collapse_map_holder['map'] = collapse_score

        logits = model(image)
        loss, parts = criterion(logits, mask, collapse_score)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())
        epoch_dices.append(1.0 - parts['dice'])

    elapsed = time.time() - t0
    mean_loss = np.mean(epoch_losses)
    mean_dice = np.mean(epoch_dices)
    history.append({'epoch': epoch, 'loss': mean_loss, 'dice': mean_dice})
    print(f'Epoch {epoch+1}/{n_epochs}  steps={len(epoch_losses)}  '
          f'mean_loss={mean_loss:.4f}  mean_dice={mean_dice:.4f}  time={elapsed:.1f}s')

hook_handle.remove()
print('\nTraining finished. Hook removed.')

In [ ]:
import torch
from torch.utils.data import DataLoader
import time

torch.manual_seed(42)
np.random.seed(42)

model.eval()
for p in model.parameters():
    p.requires_grad = False

adapter = PhysicsAwareAdapter(in_channels=32).to(device)
criterion = CollapseAwareLoss(dice_weight=1.0, bce_weight=1.0, shape_weight=0.5).to(device)
optimizer = torch.optim.Adam(adapter.parameters(), lr=2e-4)

collapse_map_holder = {'map': None}

def adapter_pre_hook(module, args):
    feat = args[0]
    collapse_map = collapse_map_holder['map']
    if collapse_map is None:
        return None
    return (adapter(feat, collapse_map),)

output_conv_module = dict(model.named_modules())['output_block.conv.conv']
hook_handle = output_conv_module.register_forward_pre_hook(adapter_pre_hook)

torch.manual_seed(123)
np.random.seed(123)
fixed_val_batches = []
attempts = 0
while len(fixed_val_batches) < 3 and attempts < 30:
    attempts += 1
    sample = train_dataset[np.random.randint(0, len(train_dataset))]
    if sample['mask'].sum().item() > 0:
        fixed_val_batches.append(sample)
print(f'Collected {len(fixed_val_batches)} fixed validation patches (with vessel present) for consistent eval.')

def evaluate_on_fixed_set():
    adapter.eval()
    dices = []
    with torch.no_grad():
        for sample in fixed_val_batches:
            image = sample['image'].unsqueeze(0).to(device)
            mask = sample['mask'].unsqueeze(0).to(device)
            collapse_score = sample['collapse_score'].unsqueeze(0).to(device)
            collapse_map_holder['map'] = collapse_score
            logits = model(image)
            probs = torch.sigmoid(logits)
            intersection = (probs * mask).sum()
            union = probs.sum() + mask.sum()
            dice = (2 * intersection + 1e-5) / (union + 1e-5)
            dices.append(dice.item())
    adapter.train()
    return np.mean(dices)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)

torch.manual_seed(42)
np.random.seed(42)
n_epochs = 8
max_steps_per_epoch = 30

adapter.train()
history = []

baseline_val_dice = evaluate_on_fixed_set()
print(f'Baseline (untrained adapter) fixed-val Dice: {baseline_val_dice:.4f}\n')

for epoch in range(n_epochs):
    epoch_losses = []
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        if step >= max_steps_per_epoch:
            break

        image = batch['image'].to(device)
        mask = batch['mask'].to(device)
        collapse_score = batch['collapse_score'].to(device)

        collapse_map_holder['map'] = collapse_score

        logits = model(image)
        loss, parts = criterion(logits, mask, collapse_score)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    val_dice = evaluate_on_fixed_set()
    elapsed = time.time() - t0
    mean_loss = np.mean(epoch_losses)
    history.append({'epoch': epoch, 'train_loss': mean_loss, 'fixed_val_dice': val_dice})
    print(f'Epoch {epoch+1}/{n_epochs}  train_loss={mean_loss:.4f}  '
          f'fixed_val_dice={val_dice:.4f}  time={elapsed:.1f}s')

hook_handle.remove()
print('\nTraining finished. Hook removed.')
print('\nIf fixed_val_dice trends upward across epochs, the adapter is genuinely learning.')
print('If it stays flat or noisy, we need to revisit LR, loss weights, or adapter capacity.')

In [ ]:
import numpy as np

total_slices = train_dataset.n_slices
split_point = int(total_slices * 0.8)

print('Total slices:', total_slices)
print('Train range: 0 to', split_point)
print('Held-out val range:', split_point, 'to', total_slices)

train_split_dataset = KidneyVesselDataset(
    '/kaggle/input/competitions/blood-vessel-segmentation/train/kidney_1_dense',
    patch_size=(32, 128, 128),
)
train_split_dataset.image_files = train_split_dataset.image_files[:split_point]
train_split_dataset.label_files = train_split_dataset.label_files[:split_point]
train_split_dataset.n_slices = split_point

val_split_dataset = KidneyVesselDataset(
    '/kaggle/input/competitions/blood-vessel-segmentation/train/kidney_1_dense',
    patch_size=(32, 128, 128),
)
val_split_dataset.image_files = val_split_dataset.image_files[split_point:]
val_split_dataset.label_files = val_split_dataset.label_files[split_point:]
val_split_dataset.n_slices = total_slices - split_point

print('\ntrain_split_dataset length:', len(train_split_dataset))
print('val_split_dataset length:', len(val_split_dataset))

np.random.seed(123)
held_out_val_patches = []
attempts = 0
while len(held_out_val_patches) < 5 and attempts < 40:
    attempts += 1
    idx = np.random.randint(0, len(val_split_dataset))
    sample = val_split_dataset[idx]
    if sample['mask'].sum().item() > 0:
        held_out_val_patches.append(sample)

print(f'\nCollected {len(held_out_val_patches)} truly held-out validation patches (never seen in training).')
for i, s in enumerate(held_out_val_patches):
    print(f'  patch {i}: vessel_voxels={s["mask"].sum().item():.0f}  collapse_max={s["collapse_score"].max().item():.3f}')

In [ ]:
import sys

check_vars = ['model', 'device', 'adapter', 'criterion', 'train_dataset',
              'compute_collapse_score', 'PhysicsAwareAdapter', 'CollapseAwareLoss',
              'KidneyVesselDataset', 'orig', 'pred']

print('Checking what is still in memory:')
for v in check_vars:
    exists = v in dir()
    print(f'  {v}: {"OK, present" if exists else "MISSING"}')

In [ ]:
import os
import subprocess
import sys

def pip(args):
    result = subprocess.run(['pip'] + args, capture_output=True, text=True)
    if result.returncode != 0:
        print('PIP STEP FAILED:', args)
        print(result.stderr[-1500:])
    return result.returncode

if not os.path.exists('/kaggle/working/vesselFM'):
    print('Cloning vesselFM...')
    ret = os.system('git clone --quiet https://github.com/bwittmann/vesselFM /kaggle/working/vesselFM')
    if ret != 0:
        raise RuntimeError('git clone failed -- check network access in notebook settings.')
else:
    print('vesselFM folder already present.')

config_path = '/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml'
assert os.path.exists(config_path), f'Config missing: {config_path}'

# pin numpy FIRST, before installing anything that might import it internally
pip(['install', '-q', '--force-reinstall', 'numpy==1.26.4'])

pip(['install', '-q', 'monai', 'hydra-core', 'omegaconf', 'huggingface_hub', 'scikit-image', 'nibabel', 'tqdm'])
pip(['install', '-q', '-e', '/kaggle/working/vesselFM', '--no-deps'])

# re-pin numpy again in case any of the above quietly upgraded it back
pip(['install', '-q', '--force-reinstall', '--no-deps', 'numpy==1.26.4'])

eval_file = '/kaggle/working/vesselFM/vesselfm/seg/utils/evaluation.py'
with open(eval_file, 'r') as f:
    content = f.read()
if 'skeletonize_3d = skeletonize' not in content:
    content = content.replace(
        'from skimage.morphology import skeletonize, skeletonize_3d',
        'from skimage.morphology import skeletonize\nskeletonize_3d = skeletonize'
    )
    with open(eval_file, 'w') as f:
        f.write(content)
    print('Patched evaluation.py for skeletonize_3d compatibility.')
else:
    print('evaluation.py already patched.')

print('\nAll installs done. IMPORTANT: restart the session now, then run ONLY the next cell (model loading) -- do not rerun this cell after restart.')

In [ ]:
import subprocess

result = subprocess.run(['pip', 'show', 'numpy'], capture_output=True, text=True)
print(result.stdout)

import numpy
print('numpy.__version__ as currently imported:', numpy.__version__)

In [ ]:
import torch, sys
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

sys.path.insert(0, '/kaggle/working/vesselFM')
import hydra
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
config_path = '/kaggle/working/vesselFM/vesselfm/seg/configs/model/dyn_unet_base.yaml'
model_cfg = OmegaConf.load(config_path)
ckpt_path = hf_hub_download(repo_id='bwittmann/vesselFM', filename='vesselFM_base.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)

model = hydra.utils.instantiate(model_cfg)
model.load_state_dict(ckpt)
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'vesselFM loaded on {device}. Total params: {n_params/1e6:.2f}M')